This notebook is focused on transforming the FRED treasury dataset to an zero-bond dataset to be used by HJM-PCA

reshape for hjm pca

In [9]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

INPUT_CSV = "historical data train.csv"
OUTPUT_CSV = "conversion reshape.csv"

MATURITIES = np.array([
    1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30
], dtype=float)

YIELD_COLS = [
    "Yield_1M", "Yield_3M", "Yield_6M", "Yield_1Y",
    "Yield_2Y", "Yield_3Y", "Yield_5Y", "Yield_7Y",
    "Yield_10Y", "Yield_20Y", "Yield_30Y"
]


def k(T):
    return round(float(T), 10)


def par_curve_to_zero_yields(par_yields_percent):
    """
    Convert one sparse Treasury CMT/par curve to continuously compounded
    zero-coupon yields at the same 11 maturities.

    Assumptions
    -----------
    1. Input yields are in percent.
    2. 1M, 3M and 6M are treated as single-payment instruments:
           P(0,T) = 1 / (1 + y*T)
    3. From 1Y onward, yields are treated as semiannual par yields.
    4. Missing semiannual par maturities are linearly interpolated.
    5. Output zero yields are continuously compounded:
           z(T) = -ln(P(0,T))/T
    """

    y = np.asarray(par_yields_percent, dtype=float) / 100.0

    if np.any(~np.isfinite(y)):
        return np.full(len(MATURITIES), np.nan)

    P = {}

    # Short end: 1M, 3M, 6M
    for T, rate in zip(MATURITIES[:3], y[:3]):
        P[k(T)] = 1.0 / (1.0 + rate * T)

    # Interpolate sparse observed par curve to every 6-month maturity
    coupon_obs_T = MATURITIES[2:]
    coupon_obs_y = y[2:]

    par_interp = interp1d(
        coupon_obs_T,
        coupon_obs_y,
        kind="linear",
        bounds_error=True
    )

    # Bootstrap discount factors on 0.5Y grid
    for T in np.arange(0.5, 30.0 + 0.5, 0.5):

        T = k(T)

        if np.isclose(T, 0.5):
            continue

        c = float(par_interp(T))

        previous_dates = np.arange(0.5, T, 0.5)

        pv_previous_coupons = sum(
            (c / 2.0) * P[k(t)]
            for t in previous_dates
        )

        P[T] = (
            1.0 - pv_previous_coupons
        ) / (
            1.0 + c / 2.0
        )

        if P[T] <= 0:
            raise ValueError(
                f"Non-positive discount factor at maturity {T} years."
            )

    zero = np.array([
        -np.log(P[k(T)]) / T
        for T in MATURITIES
    ])

    return zero * 100.0


def main():

    df = pd.read_csv(INPUT_CSV)

    # Check required columns
    required_cols = ["DATE"] + YIELD_COLS

    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Convert par yields to zero yields
    zero_matrix = np.vstack([
        par_curve_to_zero_yields(row)
        for row in df[YIELD_COLS].to_numpy()
    ])

    # Output ONLY DATE + zero-yield columns
    out = pd.DataFrame()

    out["DATE"] = df["DATE"]

    for i, col in enumerate(YIELD_COLS):
        out[col] = zero_matrix[:, i]

    # Save
    out.to_csv(OUTPUT_CSV, index=False)

    print(f"Saved: {OUTPUT_CSV}")
    print(f"Rows: {len(out):,}")
    print(f"Columns: {list(out.columns)}")

    print("\nFirst 5 rows:")
    print(out.head().to_string(index=False))


if __name__ == "__main__":
    main()

Saved: conversion reshape.csv
Rows: 4,836
Columns: ['DATE', 'Yield_1M', 'Yield_3M', 'Yield_6M', 'Yield_1Y', 'Yield_2Y', 'Yield_3Y', 'Yield_5Y', 'Yield_7Y', 'Yield_10Y', 'Yield_20Y', 'Yield_30Y']

First 5 rows:
    DATE  Yield_1M  Yield_3M  Yield_6M  Yield_1Y  Yield_2Y  Yield_3Y  Yield_5Y  Yield_7Y  Yield_10Y  Yield_20Y  Yield_30Y
1/2/2002  1.728754  1.736226  1.841496  2.269530  3.213108  3.750924  4.555274  5.041082   5.281815   6.195650   5.440437
1/3/2002  1.728754  1.726270  1.811769  2.229880  3.183548  3.710991  4.515166  5.000764   5.241467   6.168979   5.433797
1/4/2002  1.718769  1.716313  1.811769  2.239835  3.183407  3.721189  4.536111  5.044621   5.259849   6.221397   5.457973
1/7/2002  1.698797  1.676482  1.762214  2.180376  3.073557  3.611402  4.425813  4.933679   5.173461   6.095909   5.405914
1/8/2002  1.698797  1.676482  1.762214  2.180376  3.063480  3.601349  4.426472  4.934217   5.186143   6.108325   5.435565
